In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls -lah /content/drive
!ls -lah /content/drive/MyDrive

total 16K
dr-x------ 4 root root 4.0K May 26 15:47 .Encrypted
drwx------ 4 root root 4.0K May 26 15:47 MyDrive
dr-x------ 2 root root 4.0K May 26 15:47 .shortcut-targets-by-id
drwx------ 5 root root 4.0K May 26 15:47 .Trash-0
total 8.0K
drwx------ 2 root root 4.0K May 20 12:52 'Colab Notebooks'
drwx------ 2 root root 4.0K May 20 14:48 'Projet NER'


In [4]:
!ls -lah "/content/drive/MyDrive/Projet NER"

total 1.4M
-rw------- 1 root root  35K May 24 19:11 admin.jsonl
-rw------- 1 root root 284K May 20 14:24 DarNERcorp_test.csv
-rw------- 1 root root 1.1M May 20 13:56 DarNERcorp_train.csv
-rw------- 1 root root  316 May 20 13:05 docker-compose.yaml
-rw------- 1 root root  14K May 24 19:43 raw_text.txt


In [5]:
import pandas as pd

train_path = "/content/drive/MyDrive/Projet NER/DarNERcorp_train.csv"
test_path  = "/content/drive/MyDrive/Projet NER/DarNERcorp_test.csv"

def load_ner_csv(path):
    df = pd.read_csv(path)

    # enlever la colonne index vide
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # Renommer colonnes standards
    df = df.rename(columns={
        "Sentence": "sentence_id",
        "Token": "token_text_or_id",
        "Tag": "tag"
    })

    # Dans TON cas, token_text_or_id contient le texte du token (Uppsala, هيّا, ...)
    df = df.rename(columns={"token_text_or_id": "token"})

    return df[["sentence_id", "token", "tag"]]

train_df = load_ner_csv(train_path)
test_df  = load_ner_csv(test_path)

train_df.head()

,sentence_id,token,tag
0,0,Uppsala,B-LOC
1,0,),O
2,0,هيّا,O
3,0,رابع,O
4,0,أكبر,O


In [6]:
import pandas as pd

def group_sentences_to_lists(df):
    # garder l'ordre d'apparition
    df = df.copy()
    df["token"] = df["token"].astype(str)
    df["tag"] = df["tag"].astype(str)

    grouped = df.groupby("sentence_id", sort=False).agg({
        "token": list,
        "tag": list
    }).reset_index(drop=True)

    # renommer pour éviter confusion
    grouped = grouped.rename(columns={"token": "tokens", "tag": "ner_tags_str"})
    return grouped

train_grouped = group_sentences_to_lists(train_df)
test_grouped  = group_sentences_to_lists(test_df)

train_grouped.head()

,tokens,ner_tags_str
0,"[Uppsala, ), هيّا, رابع, أكبر, مدينة, ف, سّويد...","[B-LOC, O, O, O, O, O, O, B-LOC, O, O, B-LOC, ..."
1,"[ف, 2018, كان, عدد, سّكان, ديالها, 172, ،, 402...","[O, O, O, O, O, O, O, O, O, O]"
2,"[أوپيك, (, ب, لينݣليزية, :, OPEC, -, تاتعني, ل...","[B-ORG, O, O, B-MISC, O, B-ORG, O, O, B-ORG, I..."
3,"[تأسسات, هاد, لمنضمة, نهار, 14, شتنبر, 1960, ف...","[O, O, O, O, B-MISC, I-MISC, I-MISC, O, B-LOC,..."
4,"[ف, 1965, ،, رحلات, لمنضمة, ،, و, ولأّ, لمقر, ...","[O, O, O, O, O, O, O, O, O, O, O, B-LOC, O, O,..."


In [7]:
all_tags = sorted({t for seq in train_grouped["ner_tags_str"] for t in seq})
tag2id = {t:i for i,t in enumerate(all_tags)}
id2tag = {i:t for t,i in tag2id.items()}

def tags_to_ids(batch):
    batch["ner_tags"] = [[tag2id[t] for t in seq] for seq in batch["ner_tags_str"]]
    return batch

In [8]:
def has_entity(tags):
    # tags est une liste de strings: ["O", "B-LOC", ...]
    return any(t != "O" for t in tags)

In [9]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

with_ent = train_grouped[train_grouped["ner_tags_str"].apply(has_entity)]
all_o    = train_grouped[~train_grouped["ner_tags_str"].apply(has_entity)]

keep_ratio_all_o = 0.2  # 20% seulement
all_o_sample = all_o.sample(frac=keep_ratio_all_o, random_state=42)

train_grouped_rs = pd.concat([with_ent, all_o_sample], ignore_index=True)
train_grouped_rs = train_grouped_rs.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Original train sentences:", len(train_grouped))
print("With-entity sentences:", len(with_ent))
print("All-O sentences:", len(all_o))
print("Resampled train sentences:", len(train_grouped_rs))

Original train sentences: 1985
With-entity sentences: 1378
All-O sentences: 607
Resampled train sentences: 1499


In [10]:
from datasets import Dataset

full_train_ds = Dataset.from_pandas(train_grouped_rs).map(tags_to_ids, batched=True)

split = full_train_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
valid_ds = split["test"]

test_ds = Dataset.from_pandas(test_grouped).map(tags_to_ids, batched=True)

Map:   0%|          | 0/1499 [00:00<?, ? examples/s]

Map:   0%|          | 0/527 [00:00<?, ? examples/s]

In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Ignorer les tokens spéciaux ([CLS], [SEP])
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # Premier sous-morceau du mot
            else:
                # Appliquer le même label aux autres morceaux si tu veux de la robustesse,
                # ou -100 si tu veux évaluer uniquement sur le premier subtoken :
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

train_tok = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(tokenize_and_align_labels, batched=True, remove_columns=valid_ds.column_names)
test_tok  = test_ds.map(tokenize_and_align_labels, batched=True, remove_columns=test_ds.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/1349 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/527 [00:00<?, ? examples/s]

In [12]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

model_name = "xlm-roberta-large"   # <-- ajoute ça

seqeval = evaluate.load("seqeval")

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(all_tags),
    id2label=id2tag,
    label2id=tag2id
)

data_collator = DataCollatorForTokenClassification(tokenizer)

def compute_metrics(eval_pred):
    # Compatible avec (predictions, labels) OU EvalPrediction
    if isinstance(eval_pred, tuple):
        predictions, labels = eval_pred
    else:
        predictions, labels = eval_pred.predictions, eval_pred.label_ids

    preds = np.argmax(predictions, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        cur_preds, cur_labels = [], []
        for pred_id, lab_id in zip(pred_seq, label_seq):
            if lab_id == -100:
                continue
            cur_preds.append(id2tag[int(pred_id)])
            cur_labels.append(id2tag[int(lab_id)])
        true_preds.append(cur_preds)
        true_labels.append(cur_labels)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

args = TrainingArguments(
    output_dir="./results_ner_large",
    eval_strategy="epoch",
    save_strategy="epoch",          # Obligatoire pour que EarlyStopping puisse comparer les époques
    load_best_model_at_end=True,    # Charge automatiquement le modèle de l'époque 4 à la fin
    metric_for_best_model="loss",   # Se base sur la Validation Loss la plus basse
    greater_is_better=False,        # On veut que la Loss soit la plus petite possible

    num_train_epochs=8,             # Tu peux laisser 8, le callback l'arrêtera avant
    learning_rate=1.5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    fp16=True,
    optim="adamw_bnb_8bit",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Eval final sur test
trainer.evaluate(test_tok)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.524805,0.510653,0.578660,0.542534,0.846676
2,No log,0.326015,0.659855,0.779595,0.714745,0.903548
3,No log,0.268222,0.755488,0.777259,0.766219,0.919431
4,No log,0.267431,0.748070,0.830218,0.787006,0.922249
5,No log,0.256359,0.782443,0.798287,0.790285,0.929038
6,2.797802,0.255250,0.783994,0.785826,0.784909,0.924811
7,2.797802,0.287990,0.766548,0.838785,0.801041,0.927757
8,2.797802,0.293859,0.778742,0.838785,0.807649,0.928782


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.24272939562797546,
 'eval_precision': 0.7709824333116461,
 'eval_recall': 0.7596153846153846,
 'eval_f1': 0.7652567000322893,
 'eval_accuracy': 0.9360744297719088,
 'eval_runtime': 8.5681,
 'eval_samples_per_second': 61.507,
 'eval_steps_per_second': 30.812,
 'epoch': 8.0}

In [13]:
pred = trainer.predict(test_tok)
metrics = compute_metrics((pred.predictions, pred.label_ids))
metrics

{'precision': np.float64(0.7709824333116461),
 'recall': np.float64(0.7596153846153846),
 'f1': np.float64(0.7652567000322893),
 'accuracy': 0.9360744297719088}

In [14]:
from collections import Counter
cnt = Counter(t for seq in train_grouped["ner_tags_str"] for t in seq)
cnt

Counter({'B-LOC': 2222,
         'O': 45534,
         'B-ORG': 369,
         'B-MISC': 1605,
         'I-ORG': 615,
         'I-MISC': 659,
         'I-LOC': 622,
         'B-PER': 580,
         'I-PER': 518})

In [15]:
from transformers import pipeline
ner = pipeline(
    "token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)
ner("ana mchit l Casablanca m3a Karim w chrit puce Inwi")

[{'entity_group': 'PER',
  'score': np.float32(0.45595792),
  'word': 'ana',
  'start': 0,
  'end': 3},
 {'entity_group': 'LOC',
  'score': np.float32(0.9203598),
  'word': 'Casa',
  'start': 12,
  'end': 16},
 {'entity_group': 'LOC',
  'score': np.float32(0.9180817),
  'word': 'blanc',
  'start': 16,
  'end': 21},
 {'entity_group': 'LOC',
  'score': np.float32(0.8873123),
  'word': 'a',
  'start': 21,
  'end': 22},
 {'entity_group': 'LOC',
  'score': np.float32(0.686758),
  'word': 'In',
  'start': 46,
  'end': 48},
 {'entity_group': 'LOC',
  'score': np.float32(0.7117214),
  'word': 'wi',
  'start': 48,
  'end': 50}]

In [16]:
phrases_test = [
    "bghit nmsi l Rabat m3a Ahmed bch nchri carte SIM dyal Maroc Telecom",
    "Yassine saf r l Tangier lbarah w hdar m3a Orange",
    "Khwla khtat rendez-vous f l'agence Attijariwafa bank dyal Fez",
    "tla9it m3a Omar fCasablanca lbarah f l-marche",
    "fin ghadin lyouma m3a l-khout bghina nkhorjo nshorbo9ahwa"
]

# Remplace 'nlp' par le nom de ton pipeline de token classification
for p in phrases_test:
    print(f"\nTexte : {p}")
    print(f"Prédiction : {ner(p)}")


Texte : bghit nmsi l Rabat m3a Ahmed bch nchri carte SIM dyal Maroc Telecom
Prédiction : [{'entity_group': 'LOC', 'score': np.float32(0.9611136), 'word': 'Ra', 'start': 13, 'end': 15}, {'entity_group': 'LOC', 'score': np.float32(0.9638626), 'word': 'bat', 'start': 15, 'end': 18}, {'entity_group': 'PER', 'score': np.float32(0.7938177), 'word': 'Ahmed', 'start': 23, 'end': 28}, {'entity_group': 'ORG', 'score': np.float32(0.7318661), 'word': 'Maro', 'start': 54, 'end': 58}, {'entity_group': 'ORG', 'score': np.float32(0.7721542), 'word': 'c Telecom', 'start': 58, 'end': 67}]

Texte : Yassine saf r l Tangier lbarah w hdar m3a Orange
Prédiction : [{'entity_group': 'PER', 'score': np.float32(0.95109874), 'word': 'Ya', 'start': 0, 'end': 2}, {'entity_group': 'PER', 'score': np.float32(0.96985996), 'word': 's', 'start': 2, 'end': 3}, {'entity_group': 'PER', 'score': np.float32(0.921036), 'word': 'sine', 'start': 3, 'end': 7}, {'entity_group': 'PER', 'score': np.float32(0.57783204), 'word': 'Ta

In [17]:
phrases_arabes = [
    "بغيت نمشي للرباط مع أحمد باش نشري كارط سيم ديال اتصالات المغرب",
    "ياسين سافر لطنجة البارح و هضر مع أورنج",
    "خولة خذات رنديفو ف لانديكس ديال التجاري وفا بنك ف فاس",
    "تلاقيت مع عمر فكازا البارح ف المارشي",
    "فين غادين اليوم مع الخوت بغينا نخرجو نشربو قهوة"
]

for p in phrases_arabes:
    print(f"\nTexte : {p}")
    # Remplace ner(p) par ton pipeline de prediction
    print(f"Prédiction : {ner(p)}")


Texte : بغيت نمشي للرباط مع أحمد باش نشري كارط سيم ديال اتصالات المغرب
Prédiction : [{'entity_group': 'LOC', 'score': np.float32(0.8113521), 'word': 'لل', 'start': 10, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.9732824), 'word': 'ر', 'start': 12, 'end': 13}, {'entity_group': 'LOC', 'score': np.float32(0.9789393), 'word': 'باط', 'start': 13, 'end': 16}, {'entity_group': 'PER', 'score': np.float32(0.9385035), 'word': 'أحمد باش', 'start': 20, 'end': 28}, {'entity_group': 'ORG', 'score': np.float32(0.7574216), 'word': 'اتصالات المغرب', 'start': 48, 'end': 62}]

Texte : ياسين سافر لطنجة البارح و هضر مع أورنج
Prédiction : [{'entity_group': 'PER', 'score': np.float32(0.954344), 'word': 'ي', 'start': 0, 'end': 1}, {'entity_group': 'PER', 'score': np.float32(0.9236522), 'word': 'اسين', 'start': 1, 'end': 5}, {'entity_group': 'LOC', 'score': np.float32(0.97503823), 'word': 'ل', 'start': 11, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.9819701), 'word': 'طن', 'start':

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Prédiction : [{'entity_group': 'PER', 'score': np.float32(0.9652625), 'word': 'عمر فكازا', 'start': 10, 'end': 19}, {'entity_group': 'LOC', 'score': np.float32(0.9096027), 'word': 'الم', 'start': 29, 'end': 32}, {'entity_group': 'LOC', 'score': np.float32(0.86657447), 'word': 'ار', 'start': 32, 'end': 34}, {'entity_group': 'LOC', 'score': np.float32(0.93149984), 'word': 'شي', 'start': 34, 'end': 36}]

Texte : فين غادين اليوم مع الخوت بغينا نخرجو نشربو قهوة
Prédiction : []


In [18]:
import json, re
import pandas as pd

def simple_tokenize_with_offsets(text):
    """
    Tokenisation simple + offsets exacts.
    - sépare mots et ponctuation
    - retourne tokens + (start,end) char offsets
    """
    tokens = []
    offsets = []
    for m in re.finditer(r"\w+|[^\w\s]", text, flags=re.UNICODE):
        tokens.append(m.group(0))
        offsets.append((m.start(), m.end()))
    return tokens, offsets

def normalize_spans_keep_longest(spans):
    """
    spans: list of (start,end,label)
    Stratégie: trier par longueur desc, puis garder un ensemble sans chevauchement.
    """
    spans = [(int(s), int(e), lab) for s,e,lab in spans]
    spans = sorted(spans, key=lambda x: (-(x[1]-x[0]), x[0], x[1]))

    kept = []
    for s,e,lab in spans:
        overlap = False
        for ks,ke,klab in kept:
            # overlap strict
            if not (e <= ks or s >= ke):
                overlap = True
                break
        if not overlap:
            kept.append((s,e,lab))

    # re-trier par position
    kept = sorted(kept, key=lambda x: (x[0], x[1]))
    return kept

def spans_to_bio(tokens, offsets, spans):
    """
    Convertit spans (start,end,label) en BIO par token.
    """
    tags = ["O"] * len(tokens)
    spans = normalize_spans_keep_longest(spans)

    for (s,e,lab) in spans:
        first = True
        for i, (ts,te) in enumerate(offsets):
            # token intersects span
            if te <= s or ts >= e:
                continue
            prefix = "B-" if first else "I-"
            tags[i] = prefix + lab
            first = False

    return tags

def load_doccano_jsonl_to_df(jsonl_path):
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            text = ex["text"]
            spans = ex.get("label", [])  # <-- ton champ s'appelle "label"
            tokens, offsets = simple_tokenize_with_offsets(text)
            ner_tags_str = spans_to_bio(tokens, offsets, spans)
            rows.append({"text": text, "tokens": tokens, "ner_tags_str": ner_tags_str})
    return pd.DataFrame(rows)

doccano_path = "/content/drive/MyDrive/Projet NER/admin.jsonl"
df_doc = load_doccano_jsonl_to_df(doccano_path)

df_doc.head(3), df_doc.iloc[0]["tokens"], df_doc.iloc[0]["ner_tags_str"]

(                                         text  \
 0  ana mchit l Casa m3a Karim w b9ina 2 jours   
 1     salam Sara, wach ghadi l Rabat demain ?   
 2       j'ai acheté une carte SIM Inwi lyouma   
 
                                               tokens  \
 0  [ana, mchit, l, Casa, m3a, Karim, w, b9ina, 2,...   
 1  [salam, Sara, ,, wach, ghadi, l, Rabat, demain...   
 2  [j, ', ai, acheté, une, carte, SIM, Inwi, lyouma]   
 
                              ner_tags_str  
 0  [O, O, O, B-LOC, O, B-PER, O, O, O, O]  
 1     [O, B-PER, O, O, O, O, B-LOC, O, O]  
 2    [O, O, O, O, O, O, B-MISC, B-ORG, O]  ,
 ['ana', 'mchit', 'l', 'Casa', 'm3a', 'Karim', 'w', 'b9ina', '2', 'jours'],
 ['O', 'O', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'O', 'O', 'O'])

In [19]:
for i in [0, 10, 11, 27]:
    print(df_doc.iloc[i]["text"])
    print(list(zip(df_doc.iloc[i]["tokens"], df_doc.iloc[i]["ner_tags_str"])))
    print("----")

ana mchit l Casa m3a Karim w b9ina 2 jours
[('ana', 'O'), ('mchit', 'O'), ('l', 'O'), ('Casa', 'B-LOC'), ('m3a', 'O'), ('Karim', 'B-PER'), ('w', 'O'), ('b9ina', 'O'), ('2', 'O'), ('jours', 'O')]
----
j'ai envoyé email l service client Maroc Telecom
[('j', 'O'), ("'", 'O'), ('ai', 'O'), ('envoyé', 'O'), ('email', 'O'), ('l', 'O'), ('service', 'O'), ('client', 'O'), ('Maroc', 'B-ORG'), ('Telecom', 'I-ORG')]
----
le match f stade Mohammed V f Casa
[('le', 'O'), ('match', 'O'), ('f', 'O'), ('stade', 'B-LOC'), ('Mohammed', 'I-LOC'), ('V', 'I-LOC'), ('f', 'O'), ('Casa', 'B-LOC')]
----
Google Maps 3tani route l Aéroport Mohammed V
[('Google', 'B-MISC'), ('Maps', 'I-MISC'), ('3tani', 'O'), ('route', 'O'), ('l', 'O'), ('Aéroport', 'B-LOC'), ('Mohammed', 'I-LOC'), ('V', 'I-LOC')]
----


In [20]:
# --- PHASE 2: DOCCANO ---

doccano_path = "/content/drive/MyDrive/Projet NER/admin.jsonl"
df_doc = load_doccano_jsonl_to_df(doccano_path)   # fonctions convertisseur données plus haut
print(df_doc.head(2))

                                         text  \
0  ana mchit l Casa m3a Karim w b9ina 2 jours   
1     salam Sara, wach ghadi l Rabat demain ?   

                                              tokens  \
0  [ana, mchit, l, Casa, m3a, Karim, w, b9ina, 2,...   
1  [salam, Sara, ,, wach, ghadi, l, Rabat, demain...   

                             ner_tags_str  
0  [O, O, O, B-LOC, O, B-PER, O, O, O, O]  
1     [O, B-PER, O, O, O, O, B-LOC, O, O]  


In [21]:
from datasets import Dataset

doc_ds = Dataset.from_pandas(df_doc)
# Chercher la phrase contenant Inwi dans df_doc
rows = df_doc[df_doc["text"].str.contains("Inwi", case=False, na=False)]
for _, r in rows.iterrows():
    print(r["text"])
    print(list(zip(r["tokens"], r["ner_tags_str"])))
    print("----")

def doc_tags_to_ids(batch):
    batch["ner_tags"] = [[tag2id.get(t, tag2id["O"]) for t in seq] for seq in batch["ner_tags_str"]]
    return batch

doc_ds = doc_ds.map(doc_tags_to_ids, batched=True)

doc_split = doc_ds.train_test_split(test_size=0.2, seed=42)
doc_train = doc_split["train"]
doc_valid = doc_split["test"]

j'ai acheté une carte SIM Inwi lyouma
[('j', 'O'), ("'", 'O'), ('ai', 'O'), ('acheté', 'O'), ('une', 'O'), ('carte', 'O'), ('SIM', 'B-MISC'), ('Inwi', 'B-ORG'), ('lyouma', 'O')]
----
chrit puce Inwi f Casablanca
[('chrit', 'O'), ('puce', 'O'), ('Inwi', 'B-ORG'), ('f', 'O'), ('Casablanca', 'B-LOC')]
----
Inwi ma khdamach l internet daba
[('Inwi', 'B-ORG'), ('ma', 'O'), ('khdamach', 'O'), ('l', 'O'), ('internet', 'O'), ('daba', 'O')]
----
n9der nkhls facture dyal Inwi b carte bancaire ?
[('n9der', 'O'), ('nkhls', 'O'), ('facture', 'O'), ('dyal', 'O'), ('Inwi', 'B-ORG'), ('b', 'O'), ('carte', 'O'), ('bancaire', 'O'), ('?', 'O')]
----
carte SIM dyal Inwi f Marjane
[('carte', 'O'), ('SIM', 'B-MISC'), ('dyal', 'O'), ('Inwi', 'B-ORG'), ('f', 'O'), ('Marjane', 'B-ORG')]
----
Inwi kaydir 5G f Casablanca ?
[('Inwi', 'B-ORG'), ('kaydir', 'O'), ('5G', 'B-MISC'), ('f', 'O'), ('Casablanca', 'B-LOC'), ('?', 'O')]
----
Nadia bghat tbedel forfait dyal Inwi
[('Nadia', 'B-PER'), ('bghat', 'O'), ('tbedel'

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [22]:
def tokenize_and_align_labels_from_tokens(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )
    labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized.word_ids(batch_index=i)
        ex_labels = []
        for w in word_ids:
            ex_labels.append(-100 if w is None else examples["ner_tags"][i][w])
        labels.append(ex_labels)
    tokenized["labels"] = labels
    return tokenized

doc_train_tok = doc_train.map(tokenize_and_align_labels_from_tokens, batched=True, remove_columns=doc_train.column_names)
doc_valid_tok = doc_valid.map(tokenize_and_align_labels_from_tokens, batched=True, remove_columns=doc_valid.column_names)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [23]:
from transformers import TrainingArguments, Trainer

phase2_args = TrainingArguments(
    output_dir="ner_xlmr_darija_phase2_doccano",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

phase2_trainer = Trainer(
    model=trainer.model,        # <-- IMPORTANT: on repart du modèle entraîné DarNERcorp
    args=phase2_args,
    train_dataset=doc_train_tok,
    eval_dataset=doc_valid_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

phase2_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.382184,0.714286,0.748988,0.731225,0.886443
2,No log,0.304615,0.779070,0.813765,0.796040,0.910776
3,No log,0.269080,0.795367,0.834008,0.814229,0.916570


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=45, training_loss=0.3970491197374132, metrics={'train_runtime': 548.7714, 'train_samples_per_second': 1.312, 'train_steps_per_second': 0.082, 'total_flos': 32302356704352.0, 'train_loss': 0.3970491197374132, 'epoch': 3.0})

In [24]:
# Impact in-domain (DarNERcorp test)
phase2_trainer.evaluate(test_tok)

# Test code-switch
from transformers import pipeline
ner2 = pipeline("token-classification", model=phase2_trainer.model, tokenizer=tokenizer, aggregation_strategy="simple")
ner2("ana mchit l Casablanca m3a Karim w chrit puce Inwi")

[{'entity_group': 'LOC',
  'score': np.float32(0.993986),
  'word': 'Casa',
  'start': 12,
  'end': 16},
 {'entity_group': 'LOC',
  'score': np.float32(0.9927208),
  'word': 'blanc',
  'start': 16,
  'end': 21},
 {'entity_group': 'LOC',
  'score': np.float32(0.99238706),
  'word': 'a',
  'start': 21,
  'end': 22},
 {'entity_group': 'PER',
  'score': np.float32(0.9815088),
  'word': 'Karim',
  'start': 27,
  'end': 32},
 {'entity_group': 'ORG',
  'score': np.float32(0.82264733),
  'word': 'In',
  'start': 46,
  'end': 48},
 {'entity_group': 'ORG',
  'score': np.float32(0.8248241),
  'word': 'wi',
  'start': 48,
  'end': 50}]

In [25]:
phrases_test = [
    "bghit nmsi l Rabat m3a Ahmed bch nchri carte SIM dyal Maroc Telecom",
    "Yassine saf r l Tangier lbarah w hdar m3a Orange",
    "Khwla khtat rendez-vous f l'agence Attijariwafa bank dyal Fez",
    "tla9it m3a Omar fCasablanca lbarah f l-marche",
    "fin ghadin lyouma m3a l-khout bghina nkhorjo nshorbo9ahwa"
]

# Remplace 'nlp' par le nom de ton pipeline de token classification
for p in phrases_test:
    print(f"\nTexte : {p}")
    print(f"Prédiction : {ner2(p)}")


Texte : bghit nmsi l Rabat m3a Ahmed bch nchri carte SIM dyal Maroc Telecom
Prédiction : [{'entity_group': 'LOC', 'score': np.float32(0.992336), 'word': 'Ra', 'start': 13, 'end': 15}, {'entity_group': 'LOC', 'score': np.float32(0.9948718), 'word': 'bat', 'start': 15, 'end': 18}, {'entity_group': 'PER', 'score': np.float32(0.99381655), 'word': 'Ahmed', 'start': 23, 'end': 28}, {'entity_group': 'ORG', 'score': np.float32(0.947877), 'word': 'Maro', 'start': 54, 'end': 58}, {'entity_group': 'ORG', 'score': np.float32(0.90662146), 'word': 'c Telecom', 'start': 58, 'end': 67}]

Texte : Yassine saf r l Tangier lbarah w hdar m3a Orange
Prédiction : [{'entity_group': 'PER', 'score': np.float32(0.99413127), 'word': 'Ya', 'start': 0, 'end': 2}, {'entity_group': 'PER', 'score': np.float32(0.9954028), 'word': 's', 'start': 2, 'end': 3}, {'entity_group': 'PER', 'score': np.float32(0.9967673), 'word': 'sine', 'start': 3, 'end': 7}, {'entity_group': 'LOC', 'score': np.float32(0.97897243), 'word': 'Ta

In [26]:
phrases_arabes = [
    "بغيت نمشي للرباط مع أحمد باش نشري كارط سيم ديال اتصالات المغرب",
    "ياسين سافر لطنجة البارح و هضر مع أورنج",
    "خولة خذات رنديفو ف لانديكس ديال التجاري وفا بنك ف فاس",
    "تلاقيت مع عمر فكازا البارح ف المارشي",
    "فين غادين اليوم مع الخوت بغينا نخرجو نشربو قهوة"
]

for p in phrases_arabes:
    print(f"\nTexte : {p}")
    # Remplace ner(p) par ton pipeline de prediction
    print(f"Prédiction : {ner2(p)}")


Texte : بغيت نمشي للرباط مع أحمد باش نشري كارط سيم ديال اتصالات المغرب
Prédiction : [{'entity_group': 'LOC', 'score': np.float32(0.6627622), 'word': 'لل', 'start': 10, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.9882565), 'word': 'ر', 'start': 12, 'end': 13}, {'entity_group': 'LOC', 'score': np.float32(0.99305236), 'word': 'باط', 'start': 13, 'end': 16}, {'entity_group': 'PER', 'score': np.float32(0.9872579), 'word': 'أحمد', 'start': 20, 'end': 24}, {'entity_group': 'ORG', 'score': np.float32(0.93924636), 'word': 'اتصالات المغرب', 'start': 48, 'end': 62}]

Texte : ياسين سافر لطنجة البارح و هضر مع أورنج
Prédiction : [{'entity_group': 'PER', 'score': np.float32(0.99704546), 'word': 'ي', 'start': 0, 'end': 1}, {'entity_group': 'PER', 'score': np.float32(0.99788207), 'word': 'اسين', 'start': 1, 'end': 5}, {'entity_group': 'LOC', 'score': np.float32(0.95482767), 'word': 'ل', 'start': 11, 'end': 12}, {'entity_group': 'LOC', 'score': np.float32(0.9883441), 'word': 'طن', 'start'

In [27]:
trainer.save_model("/content/drive/MyDrive/Projet NER/best_darija_ner_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Projet NER/best_darija_ner_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Projet NER/best_darija_ner_model/tokenizer_config.json',
 '/content/drive/MyDrive/Projet NER/best_darija_ner_model/tokenizer.json')